# ⚽ Mission 14: Meet Your AI Soccer Coach Agent

## From AI Model to AI Agent

In Mission 13, we taught AI how to see Artin.

The computer could:
- detect players
- track Artin
- measure movement
- calculate distance
- estimate speed
- identify tactical zones

But there is an important question:
> Can AI use this information to act like a soccer coach?

That is where AI agents come in.

In this mission, we will build the first version of:
# ⚽ Artin FC AI Soccer Coach

The coach will eventually be able to:
- remember Artin
- read soccer knowledge
- analyze player data
- use software tools
- make coaching decisions
- communicate with the player

Today we build the first piece:
> **The AI Agent.**

---
## Phase 1 — What Is an AI Agent? 🤖

An **AI Agent** goes beyond simple pattern recognition or text generation. While a traditional software script follows strict, predefined rules, an AI agent combines probabilistic reasoning with goal-driven action cycles.

Key capabilities of an AI Agent include:
1. **Perception:** Interpreting incoming raw data (e.g., video frames, tracking coordinates, physical metrics).
2. **Reasoning:** Processing contextual rules, constraints, and tactical objectives.
3. **Tool Use:** Querying specialized external software (e.g., databases, optical flow trackers, feature extractors).
4. **Action:** Generating structured decisions, tactical reports, or interactive coaching feedback.

---
## Phase 2 — AI Model vs AI Agent

### AI Model
An AI model receives a prompt and generates text output based on trained statistical associations.

**Example:**
* **Player:** "What should a winger do?"
* **AI Model:** "A winger should create width, attack space, and support the attack."

This response is useful, but generic. It lacks context regarding *who* is asking, their physical capabilities, or recent performance metrics.

### AI Agent
An AI agent operates within a functional feedback system:

Goal → Reasoning → Tools → Information → Decision → Action

```
              ⚽ AI SOCCER COACH
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
           GOAL                DATA
             │                   │
             └─────────┬─────────┘
                       ↓
                   AI MODEL
                       ↓
                 DECISION
                       ↓
                 COACHING
                   ADVICE
```

**Core Distinction:** An AI model generates text. An AI agent integrates an AI model into an operational system capable of querying state data, evaluating progress against goals, and triggering software execution.

---
## Phase 3 — Build a Simple AI Coach

We initialize our base Python environment and configure model connections. In this phase, we establish client instances and set up local environment variables securely.

### 🔐 API Keys
An API key grants your application authorized access to backend LLM interfaces. **Never expose or hardcode API keys in public repositories.** Always load credentials via environment variables (`os.environ`).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve the API key from your Colab secrets securely
api_key = userdata.get('JULY-KEY')
client = genai.Client(api_key=api_key)

print("✅ AI Soccer Coach client initialized!")

---
## Phase 4 — Give the Coach a Personality

System instructions establish operational guardrails, tone, role, and domain expertise. By injecting system-level constraints, we instruct the model to adopt the role of a professional athletic coach and prohibit statistical hallucination.

In [9]:
coach_instructions = """
You are the Artin FC AI Soccer Coach.

Your job is to help a soccer player improve.

You should:
- Provide practical, high-performance soccer advice.
- Explain your tactical and physical reasoning clearly.
- Use player performance data whenever provided.
- NEVER invent or hallucinate player statistics.
- Clearly distinguish physical/tactical measurements from coaching recommendations.
- Maintain an encouraging, analytical, and professional coaching tone.
"""

print("System instructions defined successfully.")

---
## Phase 5 — Ask the First Question

We issue an initial query to test zero-shot model responses before introducing player-specific tracking telemetry.

In [ ]:

question = "How can I improve as a winger?"


response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=question,
    config=types.GenerateContentConfig(
        system_instruction=coach_instructions,
        temperature=0.2
    )
)

print(response.text)

> **Teaching Insight:** The output provides general soccer fundamentals, but lacks knowledge of Artin's specific match metrics (e.g., top speed, stamina curve, positioning heatmap). To convert this model into a contextual agent, we must feed it real metrics generated during Mission 13.

---
## Phase 6 — Bring Mission 13 Back ⚽

In Mission 13, computer vision scripts computed actual tracking parameters:
- Total Distance Covered (yd)
- Average Active Speed (yd/s)
- Maximum Sprint Speed (yd/s)
- Most Occupied Tactical Zone
- Final-Third Occupancy Percentage (%)

We construct a synthetic dataset mirroring the pandas DataFrame outputs generated in Mission 13.

In [11]:
# Reconstructing actual physical metrics from Mission 13 CV pipeline
artin_id = 4
tracking_frames_count = 240  # 8 seconds @ 30 FPS
total_distance_yards = 42.8
average_speed = 3.85  # yd/s (~7.87 mph)
maximum_speed = 7.60  # yd/s (~15.54 mph)
most_common_zone = "Final Third"
final_third_percentage = 68.5

artin_data = {
    "player": "Artin",
    "position": "Right Winger",
    "tracking_id": artin_id,
    "tracked_frames": tracking_frames_count,
    "distance_yards": round(total_distance_yards, 2),
    "average_speed": round(average_speed, 2),
    "maximum_speed": round(maximum_speed, 2),
    "most_occupied_zone": most_common_zone,
    "final_third_percentage": round(final_third_percentage, 1)
}

print("Artin Data Payload:")

#prints the contents of artin_data in a nicely formatted JSON (dictionary) style.
print(json.dumps(artin_data, indent=2))

---
## Phase 7 — Build the First Coaching Tool 🛠️

A **Tool** is a callable program function that exposes data retrieval or calculation endpoints to an execution pipeline.

In [ ]:
def get_player_statistics():
    """Returns measured performance statistics for Artin from Mission 13."""
    return artin_data

# Test tool execution
player_stats = get_player_statistics()
print("Tool output verification:", player_stats)

 `"""Returns measured performance statistics for Artin from Mission 13.""" `
 
This is called a docstring.

It explains what the function does.

It doesn't actually run the AI.

It is simply documentation for humans reading the code.

---
## Phase 8 — Let the Agent Use the Tool

We inject the output of `get_player_statistics()` directly into the agent's context prompt, forcing the LLM to ground its reasoning strictly on actual measurements.

In [ ]:
player_stats = get_player_statistics()

coach_prompt = f"""
Analyze this player's measured performance data:

Player: {player_stats['player']}
Position: {player_stats['position']}
Distance Covered: {player_stats['distance_yards']} yards
Average Speed: {player_stats['average_speed']} yards/second
Maximum Speed: {player_stats['maximum_speed']} yards/second
Most Occupied Zone: {player_stats['most_occupied_zone']}
Final-Third Occupancy: {player_stats['final_third_percentage']}%

Provide:
1. Two movement strengths based on these numbers.
2. Two physical/tactical areas to improve.
3. One tactical recommendation.
4. One training recommendation.

Do not invent statistics. Clearly distinguish measurements from coaching recommendations.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=coach_prompt,
    config=types.GenerateContentConfig(
        system_instruction=coach_instructions,
        temperature=0.2
    )
)

print(response.text)

---
## Phase 9 — Build the Agent Loop 🔄

The operational loop coordinates user intent, dynamic tool querying, state contextualization, and response synthesis:

```
        GOAL
          ↓
      UNDERSTAND
          ↓
       GET DATA  ──► (get_player_statistics)
          ↓
       ANALYZE
          ↓
       DECIDE
          ↓
       RESPOND
```

---
## Phase 10 — Build a Simple Coach Function

We encapsulate the prompt composition, tool extraction, and LLM call inside a reusable, modular function.

In [ ]:
def soccer_coach(question: str, player_data: dict) -> str:
    """
    Processes player query using verified metrics through the AI Coach Agent.
    """
    prompt = f"""
Player Information:
- Name: {player_data['player']}
- Position: {player_data['position']}
- Distance: {player_data['distance_yards']} yards
- Average Speed: {player_data['average_speed']} yd/s
- Maximum Speed: {player_data['maximum_speed']} yd/s
- Most Occupied Zone: {player_data['most_occupied_zone']}
- Final Third Occupancy: {player_data['final_third_percentage']}%

Player Question: "{question}"

Instruction: Answer directly as a professional soccer coach. Use supplied measurements accurately. Do not invent metrics.
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=coach_instructions,
            temperature=0.2
        )
    )
    return response.text

print("soccer_coach function registered successfully.")

---
## Phase 11 — Create the Coach Conversation

We test multi-turn queries against `soccer_coach()` using Artin's dataset.

In [ ]:
sample_questions = [
    "How did I perform?",
    "What was my strongest area?",
    "Where did I spend most of my time?",
    "How can I improve as a winger?",
    "What should I practice this week?"
]

In [ ]:
question = "How did I perform?"

answer = soccer_coach(question, artin_data)

print(answer)

In [ ]:
question = "What was my strongest area?"

answer = soccer_coach(question, artin_data)

print(answer)

In [ ]:
question = "What should I practice this week?"

answer = soccer_coach(question, artin_data)

print(answer)

---
## 🏆 Challenge 1 — Ask Your Coach

Evaluate whether the AI agent correctly grounds its evaluations on exact metrics (`42.8 yards`, `7.6 yd/s`, `68.5% Final Third`) without making up arbitrary statistics.

In [12]:
challenge_1_q = "Analyze my speed profile and tactical positioning. Am I pushing high enough up the pitch?"
print(soccer_coach(challenge_1_q, artin_data))

---
## 🏆 Challenge 2 — Add Another Tool

Build a secondary tool function `get_zone_analysis()` that filters positional and field-occupancy telemetry.

In [ ]:
def get_zone_analysis():
    """Tool function returning pitch location breakdown."""
    return {
        "most_occupied_zone": artin_data["most_occupied_zone"],
        "final_third_percentage": artin_data["final_third_percentage"],
        "defensive_third_percentage": 5.2,
        "midfield_percentage": 26.3
    }

zone_info = get_zone_analysis()
print("Zone Analysis Tool Output:", zone_info)

---
## ⭐ SUPER CHALLENGE — Performance Coach

Build `get_performance_report()` to aggregate physical performance metrics into a standardized report structure.

In [ ]:
def get_performance_report():
    """Generates a comprehensive player analytics summary."""
    stats = get_player_statistics()
    zones = get_zone_analysis()

    report = {
        "summary_metrics": stats,
        "spatial_distribution": zones,
        "sprint_efficiency_score": round((stats["maximum_speed"] / stats["average_speed"]), 2)
    }
    return report

def run_performance_agent(user_query):
    # Agent fetches data via tools
    full_report = get_performance_report()

    agent_prompt = f"""
Complete Performance Data:
{json.dumps(full_report, indent=2)}

User Request: {user_query}

Synthesize the report and issue an elite-level athletic analysis.
"""
    res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=agent_prompt,
        config=types.GenerateContentConfig(
            system_instruction=coach_instructions,
            temperature=0.2
        )
    )
    return res.text

print(run_performance_agent("Give me my complete performance report."))

---
## Phase 12 — First Streamlit Agent 🖥️

Below is the standalone Python script to launch the interactive **Artin FC AI Soccer Coach** web app in Streamlit.

### Streamlit Application Code (`app.py`)
Save the code block below into a file named `app.py` and run: `streamlit run app.py`

In [ ]:
import os
import json
import streamlit as st
from google import genai
from google.genai import types

st.set_page_config(page_title="Artin FC AI Soccer Coach", page_icon="⚽", layout="centered")

st.title("⚽ ARTIN FC AI SOCCER COACH")
st.caption("Mission 14 — AI Agent Interface")

# Pre-loaded player telemetry from Mission 13
artin_data = {
    "player": "Artin",
    "position": "Right Winger",
    "distance_yards": 42.8,
    "average_speed": 3.85,
    "maximum_speed": 7.60,
    "most_occupied_zone": "Final Third",
    "final_third_percentage": 68.5
}

st.sidebar.header("👤 Player Profile")
st.sidebar.markdown(f"**Player:** {artin_data['player']}")
st.sidebar.markdown(f"**Position:** {artin_data['position']}")
st.sidebar.markdown(f"**Distance:** {artin_data['distance_yards']} yards")
st.sidebar.markdown(f"**Avg Speed:** {artin_data['average_speed']} yd/s")
st.sidebar.markdown(f"**Top Speed:** {artin_data['maximum_speed']} yd/s")
st.sidebar.markdown(f"**Primary Zone:** {artin_data['most_occupied_zone']}")

user_query = st.text_input("Ask your AI Coach:", value="How did I perform in the final third?")

if st.button("Ask Coach"):
    if "GEMINI_API_KEY" not in os.environ:
        st.error("API Key not found in environment variables!")
    else:
        client = genai.Client()
        system_instruction = """
        You are the Artin FC AI Soccer Coach. Provide tactical advice grounded strictly in supplied telemetry. Never hallucinate stats.
        """
        prompt = f"""
        Player Telemetry:
        {json.dumps(artin_data, indent=2)}

        Question: {user_query}
        """
        with st.spinner("Analyzing metrics..."):
            res = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.2
                )
            )
            st.markdown("### 🤖 AI Coach Advice")
            st.write(res.text)


---
## 🏆 Mission 14 Final Challenge

Build your own version of the **Artin FC AI Soccer Coach**.

### Verification Checklist:
- [x] Accept a coaching question
- [x] Configure system instructions defining the AI Coach persona
- [x] Receive Artin's real tracking data from Mission 13
- [x] Use Python functions as tools (`get_player_statistics`, `get_zone_analysis`)
- [x] Analyze physical and tactical measurements
- [x] Generate contextual coaching advice
- [x] Strictly prohibit hallucinated statistics
- [x] Deploy interactive UI in Streamlit

---
## Final Reflection

### Question 1: What is an AI model?
*An AI model is a probabilistic algorithm trained to predict tokens or generate text/media based on learned statistical patterns.*

### Question 2: What is an AI agent?
*An AI agent is a goal-driven system that uses an AI model for reasoning while integrating tool execution, perception data, and multi-step decision loops.*

### Question 3: What is a tool?
*A tool is an executable function or API that allows an AI agent to fetch external data or trigger software actions.*

### Question 4: Why shouldn't the AI invent Artin's statistics?
*Hallucinated statistics lead to faulty coaching assessments, destroying trust and providing harmful physical or tactical guidance.*

### Question 5: Complete
- Mission 13 taught AI to **see** Artin.
- Mission 14 taught AI to **understand** Artin's data.
- Mission 15 will teach AI to **remember** Artin.

---
# 🚀 What's Next?

Our AI Coach can now analyze Artin.

But there is a problem: every time we start a new conversation, the coach forgets everything. It doesn't remember:
- Artin's previous conversations
- previous training recommendations
- previous performance
- goals
- preferences

So our next challenge is:

# 🧠 Mission 15: Give Your AI Coach a Memory

We will teach our agent how to remember.